# ESPAC DFE Inventory Interactive Plots

Interactive explorer for crop and livestock DFE inventories generated by notebook 3 workflows.

Features:
- Choose domain: crops or livestock
- Choose aggregation strategy from available DFE CSV files
- Filter by main entity (Crop/Product)
- Choose metric to plot
- Metric list is restricted to variables used in final XML inventories
- All plots are normalized per hectare (per 1 ha)
- Every plot shows uncertainty bounds using lower=min and upper=Q90

In [1]:
from __future__ import annotations

from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown
from scripts.pipeline_manifest import (
    discover_runs as manifest_discover_runs,
    latest_valid_run as manifest_latest_valid_run,
    discover_available_summary_tokens as manifest_discover_summary_tokens,
)

plt.style.use('seaborn-v0_8-whitegrid')


def find_project_dir(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / 'outputs' / 'CSVs').exists() and (p / 'notebooks').exists():
            return p
    return start


PROJECT_DIR = find_project_dir(Path.cwd().resolve())
CSVS_DIR = PROJECT_DIR / 'outputs' / 'CSVs'

DOMAIN_PREFIX = {
    'crops': 'crop',
    'livestock': 'livestock',
}

XML_DIR_BY_DOMAIN = {
    'crops': PROJECT_DIR / 'outputs' / '05_xml_exports_crop_lci',
    'livestock': PROJECT_DIR / 'outputs' / '05_xml_exports_livestock_lci',
}



def discover_strategies(domain: str) -> list[str]:
    # Manifest-first strategy discovery; fallback to legacy filename scan.
    try:
        tokens = manifest_discover_summary_tokens(PROJECT_DIR, domain)
    except Exception:
        tokens = []
    if tokens:
        return sorted(set(tokens))

    prefix = DOMAIN_PREFIX[domain]
    patt = re.compile(rf'^03-05_espac_{prefix}_lci_table_filtered_dfe__summary_(.+)\.csv$')
    out = []
    for p in CSVS_DIR.glob(f'03-05_espac_{prefix}_lci_table_filtered_dfe__summary_*.csv'):
        if p.name.endswith('_uncertainty.csv'):
            continue
        m = patt.match(p.name)
        if m:
            token = m.group(1)
            unc = p.with_name(f'{p.stem}_uncertainty.csv')
            if unc.exists():
                out.append(token)
    return sorted(set(out))


def discover_run_options(domain: str, strategy: str) -> list[tuple[str, str]]:
    rows = []
    try:
        rows = [
            r for r in manifest_discover_runs(PROJECT_DIR, domain)
            if str(r.get('summary_token', '')) == str(strategy)
            and bool(r.get('is_complete_for_notebook6'))
            and str(r.get('pipeline_stage', '')) in {'03-05', '05_xml'}
        ]
    except Exception:
        rows = []

    opts: list[tuple[str, str]] = [('Latest', '__latest__')]
    for r in rows[-20:]:
        rid = str(r.get('run_id', '')).strip()
        ts = str(r.get('created_at_utc', '')).replace('T', ' ')[:19]
        stage = str(r.get('pipeline_stage', ''))
        if rid:
            opts.append((f'{ts} | {stage} | {rid}', rid))
    return opts


def dfe_paths(domain: str, strategy: str, run_id: str | None = None) -> tuple[Path, Path]:
    # Manifest-first resolution
    record = None
    if run_id and run_id != '__latest__':
        try:
            rr = [r for r in manifest_discover_runs(PROJECT_DIR, domain) if str(r.get('run_id', '')) == str(run_id)]
            record = rr[-1] if rr else None
        except Exception:
            record = None
    else:
        try:
            record = manifest_latest_valid_run(PROJECT_DIR, domain, strategy)
        except Exception:
            record = None

    if record:
        m = Path(str(record.get('source_main_csv', '')))
        u = Path(str(record.get('source_unc_csv', '')))
        if m.exists() and u.exists():
            return m, u

    # Legacy fallback
    prefix = DOMAIN_PREFIX[domain]
    main = CSVS_DIR / f'03-05_espac_{prefix}_lci_table_filtered_dfe__summary_{strategy}.csv'
    unc = CSVS_DIR / f'03-05_espac_{prefix}_lci_table_filtered_dfe__summary_{strategy}_uncertainty.csv'
    return main, unc


def load_merged_from_run(domain: str, run_id: str) -> tuple[pd.DataFrame, list[str], str]:
    main_path, unc_path = dfe_paths(domain=domain, strategy='', run_id=run_id)
    if not main_path.exists() or not unc_path.exists():
        raise FileNotFoundError(f'Run paths missing: {main_path} | {unc_path}')

    df_main = pd.read_csv(main_path)
    df_unc = pd.read_csv(unc_path)

    keys = identity_columns(df_main, df_unc)
    if not keys:
        raise ValueError('Could not infer join keys between main and uncertainty tables.')

    metrics = metrics_with_uncertainty(df_main, df_unc)
    if not metrics:
        raise ValueError('No numeric metrics with uncertainty bounds found for this run.')

    xml_tokens = XML_METRIC_TOKENS.get(domain, set())
    if xml_tokens:
        metrics = [m for m in metrics if m in xml_tokens]
    if not metrics:
        raise ValueError('No metrics left after XML-variable filter for this run.')

    unc_cols = keys + [f'{m}__minValue' for m in metrics] + [f'{m}__maxValue' for m in metrics]
    unc_cols = [c for c in unc_cols if c in df_unc.columns]
    df_unc_small = df_unc[unc_cols].copy()

    merged = df_main.merge(df_unc_small, on=keys, how='left', suffixes=('', '_unc'))
    selector_col = primary_selector_column(domain, merged)
    return merged, metrics, selector_col


def identity_columns(df_main: pd.DataFrame, df_unc: pd.DataFrame) -> list[str]:
    cand = [
        'Region', 'Province', 'Crop', 'Category', 'Crop_group', 'Cropping_system',
        'Product', 'System', 'Species'
    ]
    keys = [c for c in cand if c in df_main.columns and c in df_unc.columns]
    if keys:
        return keys
    main_non_num = [c for c in df_main.columns if not pd.api.types.is_numeric_dtype(df_main[c])]
    return [c for c in main_non_num if c in df_unc.columns]


def primary_selector_column(domain: str, df: pd.DataFrame) -> str:
    if domain == 'crops':
        for c in ['Crop', 'Category', 'Crop_group', 'Cropping_system']:
            if c in df.columns:
                return c
    else:
        for c in ['Product', 'System', 'Species']:
            if c in df.columns:
                return c
    return df.columns[0]


def metrics_with_uncertainty(df_main: pd.DataFrame, df_unc: pd.DataFrame) -> list[str]:
    out = []
    for c in df_main.columns:
        if not pd.api.types.is_numeric_dtype(df_main[c]):
            continue
        if c.endswith('__minValue') or c.endswith('__maxValue'):
            continue
        if f'{c}__minValue' in df_unc.columns and f'{c}__maxValue' in df_unc.columns:
            out.append(c)
    return sorted(out)


def xml_metric_tokens(domain: str) -> set[str]:
    xml_root = XML_DIR_BY_DOMAIN[domain]
    if not xml_root.exists():
        return set()

    tokens: set[str] = set()
    for xml_path in xml_root.rglob('*.xml'):
        try:
            text = xml_path.read_text(encoding='utf-8', errors='ignore')
        except Exception:
            continue
        for comment in re.findall(r'generalComment="([^"]+)"', text):
            parts = [p.strip() for p in re.split(r'\s*\+\s*', comment) if p.strip()]
            for p in parts:
                if re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*', p):
                    tokens.add(p)
    return tokens


XML_METRIC_TOKENS = {
    'crops': xml_metric_tokens('crops'),
    'livestock': xml_metric_tokens('livestock'),
}


def per_ha_denominator(domain: str, df: pd.DataFrame) -> str | None:
    if domain == 'livestock':
        for c in ['Area_ha_per_1kg_product', 'Area_ha']:
            if c in df.columns:
                return c
    else:
        for c in ['Area_ha', 'Area_ha_per_1kg_product']:
            if c in df.columns:
                return c
    return None


def per_kg_denominator(domain: str, df: pd.DataFrame) -> str | None:
    candidates = [
        'Normalized_product_output_kg',
        'Reference_output_kg',
        'Live_weight_equivalent_kg',
        'Milk_output_kg_fpcm',
        'Egg_output_kg',
    ]
    for c in candidates:
        if c in df.columns:
            return c
    return None


def metric_is_per_ha(metric: str) -> bool:
    return ('_ha' in metric) and ('_per_1kg_product' not in metric)


def metric_is_per_kg_product(metric: str) -> bool:
    return '_per_1kg_product' in metric


def normalization_strategy(domain: str, df: pd.DataFrame, metric: str, norm_mode: str) -> tuple[str, str | None, str]:
    if norm_mode == 'per_1_ha':
        if metric_is_per_ha(metric):
            return 'already', None, 'per 1 ha'
        denom = per_ha_denominator(domain, df)
        if denom is not None:
            return 'divide', denom, 'per 1 ha'
        return 'unsupported', None, 'per 1 ha'

    if metric_is_per_kg_product(metric):
        return 'already', None, 'per 1 kg product'
    denom = per_kg_denominator(domain, df)
    if denom is not None:
        return 'divide', denom, 'per 1 kg product'
    return 'unsupported', None, 'per 1 kg product'


print(f'Project dir: {PROJECT_DIR}')
print(f'CSVs dir: {CSVS_DIR}')
print('Available crop strategies:', discover_strategies('crops'))
print('Available livestock strategies:', discover_strategies('livestock'))
print('XML metric tokens (crops):', len(XML_METRIC_TOKENS['crops']))
print('XML metric tokens (livestock):', len(XML_METRIC_TOKENS['livestock']))

Project dir: C:\Users\AAVADI\OneDrive - IRTA\Documents\_Proyectos\ESPAC
CSVs dir: C:\Users\AAVADI\OneDrive - IRTA\Documents\_Proyectos\ESPAC\outputs\CSVs
Available crop strategies: ['crop_group_national', 'crop_national', 'cropping_system', 'province', 'region']
Available livestock strategies: ['product', 'region']
XML metric tokens (crops): 37
XML metric tokens (livestock): 16


In [2]:
def load_merged(domain: str, strategy: str, run_id: str | None = None) -> tuple[pd.DataFrame, list[str], str]:
    main_path, unc_path = dfe_paths(domain, strategy, run_id=run_id)
    if not main_path.exists():
        raise FileNotFoundError(f'Missing main DFE CSV: {main_path}')
    if not unc_path.exists():
        raise FileNotFoundError(f'Missing uncertainty DFE CSV: {unc_path}')

    df_main = pd.read_csv(main_path)
    df_unc = pd.read_csv(unc_path)

    keys = identity_columns(df_main, df_unc)
    if not keys:
        raise ValueError('Could not infer join keys between main and uncertainty tables.')

    metrics = metrics_with_uncertainty(df_main, df_unc)
    if not metrics:
        raise ValueError('No numeric metrics with uncertainty bounds found for this strategy.')

    xml_tokens = XML_METRIC_TOKENS.get(domain, set())
    if xml_tokens:
        metrics = [m for m in metrics if m in xml_tokens]
    if not metrics:
        raise ValueError('No metrics left after XML-variable filter for this domain/strategy.')

    unc_cols = keys + [f'{m}__minValue' for m in metrics] + [f'{m}__maxValue' for m in metrics]
    unc_cols = [c for c in unc_cols if c in df_unc.columns]
    df_unc_small = df_unc[unc_cols].copy()

    merged = df_main.merge(df_unc_small, on=keys, how='left', suffixes=('', '_unc'))
    selector_col = primary_selector_column(domain, merged)
    return merged, metrics, selector_col


domain_dd = widgets.Dropdown(options=['crops', 'livestock'], value='livestock', description='Domain:')
strategy_dd = widgets.Dropdown(options=discover_strategies('livestock'), description='Strategy:')
run_dd = widgets.Dropdown(options=[('Latest', '__latest__')], value='__latest__', description='Data run:')
entity_ms = widgets.SelectMultiple(options=[], value=(), description='Entity:', rows=8)
metric_dd = widgets.Dropdown(options=[], description='Metric:')
normalization_dd = widgets.Dropdown(
    options=[('Per 1 ha', 'per_1_ha'), ('Per 1 kg product', 'per_1kg_product')],
    value='per_1_ha',
    description='Normalize:'
)
plot_mode_dd = widgets.Dropdown(
    options=['bar_with_errorbars', 'line_with_ribbon'],
    value='bar_with_errorbars',
    description='Plot mode:'
)
sort_dd = widgets.Dropdown(options=['mean_desc', 'mean_asc', 'name_asc'], value='mean_desc', description='Sort:')
topn_slider = widgets.IntSlider(value=15, min=5, max=100, step=5, description='Top N:')
show_table_cb = widgets.Checkbox(value=True, description='Show table')
status_out = widgets.Output()
plot_out = widgets.Output()

state = {
    'df': pd.DataFrame(),
    'metrics': [],
    'selector_col': None,
}


def refresh_dataset(*_):
    with status_out:
        status_out.clear_output()
        try:
            df, metrics, selector_col = load_merged(domain_dd.value, strategy_dd.value, run_id=run_dd.value)
            state['df'] = df
            state['metrics'] = metrics
            state['selector_col'] = selector_col

            metric_dd.options = metrics
            metric_dd.value = metrics[0]

            entities = sorted(df[selector_col].dropna().astype(str).unique().tolist())
            entity_ms.description = f'{selector_col}:'
            entity_ms.options = entities
            entity_ms.value = tuple(entities[: min(8, len(entities))])

            area_denom = per_ha_denominator(domain_dd.value, df)
            kg_denom = per_kg_denominator(domain_dd.value, df)
            print(f'Loaded {len(df):,} rows for {domain_dd.value} / {strategy_dd.value}')
            print(f'Join selector: {selector_col}')
            print(f'Metrics with uncertainty and present in final XMLs: {len(metrics)}')
            print(f'Per-ha denominator candidate: {area_denom}')
            print(f'Per-kg-product denominator candidate: {kg_denom}')
        except Exception as e:
            state['df'] = pd.DataFrame()
            state['metrics'] = []
            state['selector_col'] = None
            metric_dd.options = []
            entity_ms.options = []
            entity_ms.value = ()
            print(f'Error: {e}')


def refresh_strategy_options(*_):
    opts = discover_strategies(domain_dd.value)
    strategy_dd.options = opts
    strategy_dd.value = opts[0] if opts else None
    refresh_run_options()


def refresh_run_options(*_):
    opts = discover_run_options(domain_dd.value, strategy_dd.value)
    run_dd.options = opts
    run_dd.value = '__latest__'

domain_dd.observe(refresh_strategy_options, names='value')
strategy_dd.observe(refresh_run_options, names='value')
strategy_dd.observe(refresh_dataset, names='value')
run_dd.observe(refresh_dataset, names='value')

refresh_strategy_options()
refresh_dataset()



In [3]:
def render_plot(*_):
    with plot_out:
        plot_out.clear_output()

        df = state['df']
        selector_col = state['selector_col']
        metric = metric_dd.value
        if df.empty or selector_col is None or metric is None:
            display(Markdown('No data to plot.'))
            return

        data = df.copy()
        selected = list(entity_ms.value)
        if selected:
            data = data[data[selector_col].astype(str).isin(selected)]

        cols_needed = [selector_col, metric, f'{metric}__minValue', f'{metric}__maxValue']
        missing = [c for c in cols_needed if c not in data.columns]
        if missing:
            display(Markdown(f'Missing required columns for uncertainty plot: {missing}'))
            return

        norm_mode = normalization_dd.value
        action, denom_col, norm_label = normalization_strategy(domain_dd.value, data, metric, norm_mode)
        if action == 'unsupported':
            display(Markdown(f'Cannot normalize metric `{metric}` to {norm_label} with currently available denominator columns.'))
            return

        if action == 'divide':
            w = data[[selector_col, metric, f'{metric}__minValue', f'{metric}__maxValue', denom_col]].copy()
            w.columns = ['label', 'mean', 'min', 'max', 'denom']
        else:
            w = data[[selector_col, metric, f'{metric}__minValue', f'{metric}__maxValue']].copy()
            w.columns = ['label', 'mean', 'min', 'max']
            w['denom'] = 1.0

        w['label'] = w['label'].astype(str)
        w['mean'] = pd.to_numeric(w['mean'], errors='coerce')
        w['min'] = pd.to_numeric(w['min'], errors='coerce')
        w['max'] = pd.to_numeric(w['max'], errors='coerce')
        w['denom'] = pd.to_numeric(w['denom'], errors='coerce')
        w = w.dropna(subset=['mean', 'denom'])
        w = w[w['denom'] > 0]
        if w.empty:
            display(Markdown('No plottable rows after filtering and normalization requirements.'))
            return

        # Apply requested normalization.
        w['mean'] = w['mean'] / w['denom']
        w['min'] = w['min'] / w['denom']
        w['max'] = w['max'] / w['denom']

        # Keep lower bound coherent and inclusive around mean.
        w['min'] = w[['min', 'mean']].min(axis=1).fillna(w['mean'])

        # Always use Q90 as uncertainty upper bound (computed on current filtered plot data).
        q90_upper = pd.to_numeric(w['max'], errors='coerce').quantile(0.90)
        if pd.isna(q90_upper):
            q90_upper = pd.to_numeric(w['max'], errors='coerce').max()

        w['max'] = pd.to_numeric(w['max'], errors='coerce').fillna(w['mean'])
        w['max'] = np.minimum(w['max'], q90_upper)
        w['max'] = np.maximum(w['max'], w['mean'])

        if sort_dd.value == 'mean_desc':
            w = w.sort_values('mean', ascending=False)
        elif sort_dd.value == 'mean_asc':
            w = w.sort_values('mean', ascending=True)
        else:
            w = w.sort_values('label', ascending=True)

        w = w.head(topn_slider.value).reset_index(drop=True)
        err_low = (w['mean'] - w['min']).clip(lower=0).to_numpy()
        err_high = (w['max'] - w['mean']).clip(lower=0).to_numpy()

        metric_label = f'{metric} ({norm_label})'
        plot_mode = plot_mode_dd.value

        if plot_mode == 'bar_with_errorbars':
            fig_h = max(4.5, 0.35 * len(w) + 1.5)
            fig, ax = plt.subplots(figsize=(11, fig_h))
            y = np.arange(len(w))

            ax.barh(
                y,
                w['mean'].to_numpy(),
                xerr=np.vstack([err_low, err_high]),
                capsize=3,
                alpha=0.85,
                color='#2a9d8f',
                ecolor='#264653'
            )
            ax.set_yticks(y)
            ax.set_yticklabels(w['label'])
            ax.invert_yaxis()
            ax.set_xlabel(metric_label)
            ax.set_title(f'{domain_dd.value} | strategy={strategy_dd.value} | mode=bar | uncertainty: min-Q90 | {norm_label}')
            plt.tight_layout()
            plt.show()
        else:
            fig_w = max(11, 0.45 * len(w))
            fig, ax = plt.subplots(figsize=(fig_w, 5.8))
            x = np.arange(len(w))
            mean = w['mean'].to_numpy()
            lo = w['min'].to_numpy()
            hi = w['max'].to_numpy()

            ax.plot(x, mean, color='#264653', linewidth=2, marker='o', markersize=4, label='mean')
            ax.fill_between(x, lo, hi, color='#2a9d8f', alpha=0.25, label='uncertainty [min, Q90]')
            ax.set_xticks(x)
            ax.set_xticklabels(w['label'], rotation=65, ha='right')
            ax.set_ylabel(metric_label)
            ax.set_title(f'{domain_dd.value} | strategy={strategy_dd.value} | mode=line+ribbon | uncertainty: min-Q90 | {norm_label}')
            ax.legend(loc='best')
            plt.tight_layout()
            plt.show()

        if show_table_cb.value:
            display(Markdown(f'### Data used in plot ({norm_label})'))
            display(w[['label', 'mean', 'min', 'max']])


for w in [entity_ms, metric_dd, normalization_dd, plot_mode_dd, sort_dd, topn_slider, show_table_cb]:
    w.observe(render_plot, names='value')

controls = widgets.VBox([
    widgets.HBox([domain_dd, strategy_dd]),
    entity_ms,
    widgets.HBox([metric_dd, normalization_dd]),
    widgets.HBox([plot_mode_dd, sort_dd, topn_slider, show_table_cb]),
])

display(Markdown('## Interactive DFE Plot Explorer'))
display(controls)
display(status_out)
display(plot_out)
render_plot()

## Interactive DFE Plot Explorer

Output()

Output()

In [4]:
# Interactive LCI table explorer (copy/paste friendly)
from IPython.display import HTML, Javascript
import xml.etree.ElementTree as ET
import json
import sqlite3
import unicodedata


def _to_verbose_label(name: str) -> str:
    txt = str(name).strip()
    txt = txt.replace('__', ' - ').replace('_', ' ')
    txt = re.sub(r'\s+', ' ', txt).strip()
    return txt


def _seed_label_suffix_for_groupby(groupby_col: str) -> str:
    g = str(groupby_col or '').strip().lower()
    if g == 'crop':
        return 'crop'
    if g in {'crop_group', 'crop_group_national'}:
        return 'crop group'
    return 'crop group'


def _format_emission_compartment_label(label: str) -> str:
    m = re.match(r'^(.*)\s\[(air|water|soil|other)\]$', str(label).strip(), flags=re.I)
    if not m:
        return label
    base = m.group(1).strip()
    comp = m.group(2).lower()
    return f"{base} --> {comp}"


def _final_display_label(label: str) -> str:
    txt = str(label).strip()
    if txt == 'SOC mean rate kgChayr':
        return 'SOC turnover (kg C/ha/yr)'
    if re.fullmatch(r'Area\s+ha', txt, flags=re.I):
        return 'Area (ha)'

    # Remove verbose unit suffixes from labels for cleaner display.
    txt = re.sub(r'\bper\s*1\s*kg\s*product\b', '', txt, flags=re.I)
    txt = re.sub(r'\bper\s*1kg\s*product\b', '', txt, flags=re.I)
    txt = re.sub(r'\bkgha\b', '', txt, flags=re.I)
    txt = re.sub(r'\bpasture\s+feed\b', 'forages pastures', txt, flags=re.I)
    txt = re.sub(r'\bunmatched\s+pasture\s+feed\b', 'unmatched forages pastures', txt, flags=re.I)
    txt = re.sub(r'forages\s+pastures\s*(?:kg)?\s*\+\s*unmatched\s+forages\s+pastures\s*(?:kg)?', 'Forages pastures', txt, flags=re.I)
    txt = re.sub(r'\bkg\b', '', txt, flags=re.I)
    txt = re.sub(r'\bwater\s+l\b', 'Water (L)', txt, flags=re.I)
    txt = re.sub(r'\belectricity\s*kwh\b', 'Electricity (kWh)', txt, flags=re.I)
    txt = re.sub(r'\banimals\s+total\s+live\s+weight\s*kg\b', 'Animals total live weight', txt, flags=re.I)
    txt = re.sub(r'\s{2,}', ' ', txt).strip()
    if re.fullmatch(r'forages\s+pastures', txt, flags=re.I):
        txt = 'Forages pastures'
    txt = txt.replace(' ()', '')
    txt = re.sub(r'\s+\+\s+', ' + ', txt).strip()
    return txt


def _format_table_for_display(table: pd.DataFrame) -> pd.DataFrame:
    if table.empty:
        return table

    def _fmt(v, sci: bool):
        val = pd.to_numeric(v, errors='coerce')
        if pd.isna(val):
            return ''
        if float(val) == 0.0:
            return '-'
        return f"{val:.2e}" if sci else f"{val:.2f}"

    out_rows = []
    out_index = []
    for idx, row in table.iterrows():
        idx_l = str(idx).lower()
        is_emission = ('--> air' in idx_l) or ('--> water' in idx_l) or ('--> soil' in idx_l) or ('--> other' in idx_l)
        is_feed = ('feed' in idx_l) or ('forages pastures' in idx_l)
        is_area = bool(re.fullmatch(r'area\s*\(ha\)|area\s+ha', idx_l, flags=re.I))
        if idx_l == 'n':
            out_rows.append([
                '' if pd.isna(pd.to_numeric(v, errors='coerce')) else str(int(round(float(pd.to_numeric(v, errors='coerce')))))
                for v in row.to_list()
            ])
        else:
            out_rows.append([_fmt(v, is_emission or is_feed or is_area) for v in row.to_list()])
        out_index.append(idx)

    return pd.DataFrame(out_rows, index=out_index, columns=table.columns)


def _local_name(tag: str) -> str:
    return str(tag).split('}', 1)[1] if '}' in str(tag) else str(tag)


def _candidate_filter_columns(df: pd.DataFrame) -> list[str]:
    preferred = [
        'Region', 'Province', 'Category', 'Crop_group', 'crop_group_national', 'Cropping_system',
        'System', 'Species', 'Product', 'Crop'
    ]
    cols = []
    for c in preferred:
        if c not in df.columns:
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            continue
        nunique = df[c].dropna().astype(str).nunique()
        if c == 'Crop':
            if nunique <= 200:
                cols.append(c)
            continue
        if 1 < nunique <= 200:
            cols.append(c)

    for c in df.columns:
        if c in cols or pd.api.types.is_numeric_dtype(df[c]):
            continue
        nunique = df[c].dropna().astype(str).nunique()
        if 1 < nunique <= 200:
            cols.append(c)
    return cols


def _build_scrollable_html(df: pd.DataFrame, max_h: int = 620) -> str:
    if df.empty:
        return '<div>No rows to display.</div>'
    html_table = df.to_html(index=True, escape=False)
    return f"""
    <div style='max-height:{max_h}px; overflow:auto; border:1px solid #ddd; padding:6px;'>
      {html_table}
    </div>
    """


def _emission_medium(category: str, subcategory: str) -> str:
    txt = f"{category or ''} {subcategory or ''}".lower()
    if 'air' in txt:
        return 'air'
    if 'water' in txt:
        return 'water'
    if 'soil' in txt or 'ground' in txt:
        return 'soil'
    return 'other'


def _exchange_display_key(general_comment: str, name: str, input_group: str, output_group: str, category: str, subcategory: str) -> str:
    # For emissions use pollutant identity instead of factor-code comments.
    if output_group == '4':
        medium = _emission_medium(category, subcategory)
        pollutant = str(name or '').strip() or str(general_comment or '').strip()
        if pollutant:
            return f"{pollutant} [{medium}]"

    gc = str(general_comment or '').strip()
    nm = str(name or '').strip()
    return gc if gc else nm


def _section_rank(input_group: str, output_group: str, number: str, category: str, subcategory: str) -> int:
    if output_group == '0' or pd.to_numeric(number, errors='coerce') == 1:
        return 1  # reference flow
    if input_group == '4':
        return 2  # inputs from nature
    if input_group == '5':
        return 3  # inputs from technosphere
    if output_group == '4':
        med = _emission_medium(category, subcategory)
        if med == 'air':
            return 4
        if med == 'water':
            return 5
        if med == 'soil':
            return 6
        return 7
    return 8


def _strategy_xml_dir(domain: str, strategy: str) -> Path:
    return XML_DIR_BY_DOMAIN[domain] / f'summary_{strategy}'


def _reference_flow_key(domain: str) -> str:
    return "reference flow (1 ha)" if domain == 'crops' else "reference flow (1 kg)"


def _is_reference_exchange(input_group: str, output_group: str, number: str) -> bool:
    return (output_group == '0') or (pd.to_numeric(number, errors='coerce') == 1)


def _parse_xml_inventory(xml_path: Path, domain: str) -> list[dict]:
    root = ET.parse(xml_path).getroot()
    rows = []
    seq = 0
    for ex in (e for e in root.iter() if _local_name(e.tag) == 'exchange'):
        seq += 1
        gc = str(ex.attrib.get('generalComment', '') or '').strip()
        if not gc:
            gc = str(ex.attrib.get('name', '') or '').strip()
        if not gc:
            continue

        inp, out = '', ''
        for ch in ex:
            ln = _local_name(ch.tag)
            tx = str(ch.text or '').strip()
            if ln == 'inputGroup':
                inp = tx
            elif ln == 'outputGroup':
                out = tx

        ex_number_raw = str(ex.attrib.get('number', '') or '')
        section = _section_rank(
            input_group=inp,
            output_group=out,
            number=ex_number_raw,
            category=str(ex.attrib.get('category', '') or ''),
            subcategory=str(ex.attrib.get('subCategory', '') or ''),
        )
        number_rank = pd.to_numeric(ex.attrib.get('number', None), errors='coerce')
        number_rank = int(number_rank) if pd.notna(number_rank) else 999999
        mean_val = pd.to_numeric(ex.attrib.get('meanValue', None), errors='coerce')

        if _is_reference_exchange(inp, out, ex_number_raw):
            key = _reference_flow_key(domain)
        else:
            key = _exchange_display_key(
                general_comment=gc,
                name=str(ex.attrib.get('name', '') or ''),
                input_group=inp,
                output_group=out,
                category=str(ex.attrib.get('category', '') or ''),
                subcategory=str(ex.attrib.get('subCategory', '') or ''),
            )

        rows.append({
            'exchange_key': key,
            'mean_value': float(mean_val) if pd.notna(mean_val) else np.nan,
            'section_rank': section,
            'number_rank': number_rank,
            'seq_rank': seq,
        })
    return rows


def _build_xml_long_table(domain: str, strategy: str, df_summary: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, tuple[int,int,int]]]:
    xml_dir = _strategy_xml_dir(domain, strategy)
    if not xml_dir.exists():
        raise FileNotFoundError(f'Missing XML strategy folder: {xml_dir}')

    xml_files = sorted(xml_dir.glob('*.xml'))
    if not xml_files:
        raise FileNotFoundError(f'No XML files found in: {xml_dir}')

    # Prefer explicit 00001_, 00002_, ... mapping when present.
    # Fallback: some exports (e.g. livestock) use descriptive names without numeric prefixes;
    # in that case, map files to summary rows by sorted position.
    file_to_row_idx: list[tuple[Path, int]] = []
    numeric_matches = [re.match(r'^(\d+)_', xp.name) for xp in xml_files]
    if all(m is not None for m in numeric_matches):
        for xp, m in zip(xml_files, numeric_matches):
            row_no = int(m.group(1))
            file_to_row_idx.append((xp, row_no - 1))
    else:
        file_to_row_idx = [(xp, i) for i, xp in enumerate(xml_files)]

    long_rows = []
    order_map: dict[str, tuple[int,int,int]] = {}

    for xp, row_idx in file_to_row_idx:
        if row_idx < 0 or row_idx >= len(df_summary):
            continue

        meta = df_summary.iloc[row_idx]
        exchanges = _parse_xml_inventory(xp, domain)
        for ex in exchanges:
            rec = {k: meta.get(k, np.nan) for k in df_summary.columns}
            rec['exchange_key'] = ex['exchange_key']
            rec['mean_value'] = ex['mean_value']
            long_rows.append(rec)

            cand = (ex['section_rank'], ex['number_rank'], ex['seq_rank'])
            prev = order_map.get(ex['exchange_key'])
            if prev is None or cand < prev:
                order_map[ex['exchange_key']] = cand

    if not long_rows:
        raise ValueError('No XML exchanges could be mapped to summary rows.')

    long_df = pd.DataFrame(long_rows)
    return long_df, order_map


def _drop_all_zero_or_na_rows(wide: pd.DataFrame) -> pd.DataFrame:
    if wide.empty:
        return wide
    num = wide.apply(pd.to_numeric, errors='coerce')
    keep_mask = ~(num.fillna(0) == 0).all(axis=1)
    # Keep land-use transformation/occupation rows even when zeros so the structure is explicit.
    force_keep_re = re.compile(r'(occupation|transformation)', re.I)
    forced_idx = [idx for idx in wide.index if force_keep_re.search(str(idx))]
    keep_mask.loc[forced_idx] = True
    return wide.loc[keep_mask]


def _drop_systematic_zero_pesticide_rows(wide: pd.DataFrame) -> pd.DataFrame:
    # Keep pesticide-related rows whenever at least one numeric value is non-zero.
    pest_re = re.compile(r'(pesticide|herbicide|fungicide|insecticide|biocide|pest)', re.I)
    keep = []
    for idx, row in wide.iterrows():
        if not pest_re.search(str(idx)):
            keep.append(idx)
            continue
        vals = pd.to_numeric(row, errors='coerce').fillna(0)
        if not (vals == 0).all():
            keep.append(idx)
    return wide.loc[keep]


def _drop_redundant_metadata_rows(wide: pd.DataFrame, domain: str) -> pd.DataFrame:
    # Remove repeated descriptive reference-comment rows that are not exchange variables.
    # Keep the normalized consolidated reference row.
    keep = []
    for idx in wide.index:
        key = str(idx)
        key_l = key.lower().strip()
        if key == _reference_flow_key(domain):
            keep.append(idx)
            continue
        if key_l.startswith('model annual crops,'):
            continue
        if key_l.startswith('model_annual_crops,'):
            continue
        keep.append(idx)
    return wide.loc[keep]


domain_tbl_dd = widgets.Dropdown(options=['crops', 'livestock'], value='livestock', description='Domain:')
strategy_tbl_dd = widgets.Dropdown(options=[], description='Strategy:')
run_tbl_dd = widgets.Dropdown(options=[('Latest', '__latest__')], value='__latest__', description='Data run:')
agg_tbl_dd = widgets.Dropdown(options=[('Median', 'median'), ('Mean', 'mean'), ('Sum', 'sum')], value='median', description='Aggregate:')
status_tbl_out = widgets.Output()
result_tbl_out = widgets.Output()

group_values_ms = widgets.SelectMultiple(options=[], value=(), description='Groups:', rows=12)

# Crops controls (mirror notebook 2 crops explorer)
crops_region_filter = widgets.Dropdown(options=['All'], value='All', description='Region:', layout=widgets.Layout(width='220px'))
crops_category_filter = widgets.Dropdown(options=['All'], value='All', description='Category:', layout=widgets.Layout(width='220px'))
crops_summary_filter = widgets.Dropdown(options=[], description='Summary:', layout=widgets.Layout(width='320px'))
crops_province_filter = widgets.SelectMultiple(options=tuple(), value=tuple(), description='Provinces:', rows=8, layout=widgets.Layout(width='320px'))
crops_crop_filter = widgets.Dropdown(options=['All'], value='All', description='Crop:', layout=widgets.Layout(width='320px'))
crops_subcrop_filter = widgets.Dropdown(options=['All'], value='All', description='Sub-crop:', layout=widgets.Layout(width='320px'))
crops_subcrop_filter.layout.display = 'none'

# Livestock controls (mirror notebook 2 livestock explorer)
livestock_region_filter = widgets.Dropdown(options=['All'], value='All', description='Region:', layout=widgets.Layout(width='220px'))
livestock_summary_filter = widgets.Dropdown(options=[], description='Summary:', layout=widgets.Layout(width='360px'))
livestock_province_filter = widgets.SelectMultiple(options=tuple(), value=tuple(), description='Provinces:', rows=8, layout=widgets.Layout(width='320px'))
livestock_product_filter = widgets.SelectMultiple(options=tuple(), value=tuple(), description='Products:', rows=8, layout=widgets.Layout(width='240px'))
livestock_species_filter = widgets.SelectMultiple(options=tuple(), value=tuple(), description='Species:', rows=8, layout=widgets.Layout(width='220px'))
livestock_system_filter = widgets.SelectMultiple(options=tuple(), value=tuple(), description='Systems:', rows=8, layout=widgets.Layout(width='260px'))

crops_controls_box = widgets.VBox()
livestock_controls_box = widgets.VBox()
copy_html_btn = widgets.Button(description='Copy HTML', button_style='')
copy_csv_btn = widgets.Button(description='Copy CSV', button_style='')
copy_tsv_btn = widgets.Button(description='Copy TSV', button_style='')
export_tbl_out = widgets.Output()

state_tbl = {
    'summary_df': pd.DataFrame(),
    'xml_long_df': pd.DataFrame(),
    'order_map': {},
    'filter_widgets': {},
    'strategy_sync_locked': False,
    'callbacks_locked': False,
    'last_table': pd.DataFrame(),
    'last_table_html': '',
}


def _refresh_run_table_options(*_):
    opts = discover_run_options(domain_tbl_dd.value, strategy_tbl_dd.value)
    run_tbl_dd.options = opts
    run_tbl_dd.value = '__latest__'


def _refresh_strategy_table_options(*_):
    domain = domain_tbl_dd.value
    active_tokens = discover_strategies(domain)
    strategy_tbl_dd.options = active_tokens
    strategy_tbl_dd.value = active_tokens[0] if active_tokens else None

    crop_tokens = discover_strategies('crops')
    livestock_tokens = discover_strategies('livestock')

    state_tbl['strategy_sync_locked'] = True
    try:
        crops_summary_filter.options = _map_strategy_options('crops', crop_tokens)
        livestock_summary_filter.options = _map_strategy_options('livestock', livestock_tokens)

        _refresh_run_table_options()

        if domain == 'crops' and strategy_tbl_dd.value is not None:
            crop_values = [v for _, v in crops_summary_filter.options]
            if strategy_tbl_dd.value in crop_values:
                crops_summary_filter.value = strategy_tbl_dd.value
        if domain == 'livestock' and strategy_tbl_dd.value is not None:
            livestock_values = [v for _, v in livestock_summary_filter.options]
            if strategy_tbl_dd.value in livestock_values:
                livestock_summary_filter.value = strategy_tbl_dd.value
    finally:
        state_tbl['strategy_sync_locked'] = False


def _sorted_unique(df: pd.DataFrame, col: str) -> tuple:
    if col not in df.columns:
        return tuple()
    return tuple(sorted(df[col].dropna().astype(str).unique().tolist()))



OTROS_CROP_LABELS = {'OTROS PERMANENTES', 'OTROS TRANSITORIOS'}

COSTA_PROVINCES = {
    'ESMERALDAS', 'MANABI', 'LOS RIOS', 'GUAYAS', 'SANTA ELENA', 'EL ORO', 'SANTO DOMINGO DE LOS TSACHILAS'
}
SIERRA_PROVINCES = {
    'AZUAY', 'BOLIVAR', 'CANAR', 'CARCHI', 'CHIMBORAZO', 'COTOPAXI', 'IMBABURA', 'LOJA', 'PICHINCHA', 'TUNGURAHUA'
}
ORIENTE_PROVINCES = {
    'MORONA SANTIAGO', 'NAPO', 'ORELLANA', 'PASTAZA', 'SUCUMBIOS', 'ZAMORA CHINCHIPE'
}

DB_CANDIDATES = [
    PROJECT_DIR / 'outputs' / '01_espac_2024.sqlite',
    PROJECT_DIR / 'outputs' / 'CSVs' / '01_espac_2024.sqlite',
    PROJECT_DIR / 'outputs' / 'espac_2024.sqlite',
]


def _count_tables(db_path: Path) -> int:
    try:
        with sqlite3.connect(db_path) as conn:
            q = pd.read_sql_query("SELECT COUNT(*) AS n FROM sqlite_master WHERE type='table'", conn)
        return int(q.loc[0, 'n'])
    except Exception:
        return -1


def _resolve_db_path() -> Path | None:
    existing = [p for p in DB_CANDIDATES if p.exists()]
    if not existing:
        return None
    return max(existing, key=_count_tables)


DB_PATH = _resolve_db_path()


def _normalize_geo_text(text: str) -> str:
    txt = str(text).strip().upper()
    txt = unicodedata.normalize('NFKD', txt)
    txt = ''.join(ch for ch in txt if not unicodedata.combining(ch))
    txt = re.sub(r'\s+', ' ', txt)
    return txt


def map_provincia_to_region(provincia: str) -> str:
    p = _normalize_geo_text(provincia)
    if p in COSTA_PROVINCES:
        return 'costa'
    if p in SIERRA_PROVINCES:
        return 'sierra'
    if p in ORIENTE_PROVINCES:
        return 'oriente'
    return '(sin region)'


def _resolve_first_existing_table(conn: sqlite3.Connection, candidates: list[str]) -> str | None:
    names = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)['name'].astype(str).tolist()
    names_set = set(names)
    for c in candidates:
        if c in names_set:
            return c
    return None


def _load_otros_subcrop_rows(crop_label: str) -> pd.DataFrame:
    crop_label = str(crop_label).strip().upper()
    if crop_label not in OTROS_CROP_LABELS or DB_PATH is None:
        return pd.DataFrame(columns=['identificador', 'provincia', 'subcrop'])

    with sqlite3.connect(DB_PATH) as conn:
        enc_tbl = _resolve_first_existing_table(conn, ['encuestas'])
        if enc_tbl is None:
            return pd.DataFrame(columns=['identificador', 'provincia', 'subcrop'])

        if crop_label == 'OTROS PERMANENTES':
            src_tbl = _resolve_first_existing_table(conn, ['rel_inec_cpnac', 'inec_cpnac'])
            crop_col = 'cp_nclavr'
            detail_candidates = ['rc_clacul']
        else:
            src_tbl = _resolve_first_existing_table(conn, ['rel_inec_ctnac', 'inec_ctnac'])
            crop_col = 'ct_nclavr'
            detail_candidates = ['ct_codcultiv1_int', 'rc_clacul', 'ct_codcultiv2_int']

        if src_tbl is None:
            return pd.DataFrame(columns=['identificador', 'provincia', 'subcrop'])

        src_cols = pd.read_sql_query(f"PRAGMA table_info('{src_tbl}')", conn)['name'].astype(str).tolist()
        detail_col = next((c for c in detail_candidates if c in src_cols), None)
        if detail_col is None:
            return pd.DataFrame(columns=['identificador', 'provincia', 'subcrop'])

        q = f"""
            SELECT
                CAST(s.identificador AS TEXT) AS identificador,
                e.provincia AS provincia,
                TRIM(COALESCE(NULLIF(s.{detail_col}, ''), NULLIF(s.{crop_col}, ''))) AS subcrop
            FROM "{src_tbl}" s
            JOIN "{enc_tbl}" e
              ON CAST(e.identificador AS TEXT) = CAST(s.identificador AS TEXT)
            WHERE UPPER(TRIM(COALESCE(s.{crop_col}, ''))) = ?
        """
        raw = pd.read_sql_query(q, conn, params=[crop_label])

    if raw.empty:
        return raw

    raw['provincia'] = raw['provincia'].astype(str)
    raw['Region'] = raw['provincia'].map(map_provincia_to_region)
    return raw


def _list_otros_subcrops(crop_label: str) -> list[str]:
    crop_label = str(crop_label).strip().upper()
    if crop_label not in OTROS_CROP_LABELS:
        return []

    raw = _load_otros_subcrop_rows(crop_label)
    if raw.empty:
        return []

    if crops_region_filter.value != 'All':
        raw = raw[raw['Region'] == crops_region_filter.value]
    if crops_province_filter.value:
        raw = raw[raw['provincia'].isin(list(crops_province_filter.value))]

    sub = raw['subcrop'].astype(str).str.strip()
    sub = sub[(sub != '') & (sub.str.upper() != crop_label)]

    # Keep all OTROS sub-crops from DB (even if names overlap with top-level crops),
    # to match notebook-2 expectation for OTROS expansion.
    return sorted(sub.unique().tolist())


CROP_SUMMARY_LEVEL_OPTIONS = [
    ('By province', 'province'),
    ('By region (confounded provinces)', 'region'),
    ('By crop, national (confounded regions and provinces)', 'crop_national'),
    ('By cropping system (monocrop/in association, confounded provinces)', 'cropping_system'),
    ('By irrigation class (Irrig_m3 = 0 / <> 0, confounded provinces)', 'irrig_m3_class'),
    ('By farm size class (confounded provinces)', 'farm_size_class'),
    ('By crop group (confounded provinces)', 'crop_group'),
    ('By crop group, national (confounded regions and provinces)', 'crop_group_national'),
]

LIVESTOCK_SUMMARY_LEVEL_OPTIONS = [
    ('By province', 'province'),
    ('By region (confounded provinces)', 'region'),
    ('By product, national (all regions/provinces/systems combined)', 'product'),
    ('By product + system, national (confounded regions and provinces)', 'national'),
]


def _map_strategy_options(domain: str, available_tokens: list[str]):
    base = CROP_SUMMARY_LEVEL_OPTIONS if domain == 'crops' else LIVESTOCK_SUMMARY_LEVEL_OPTIONS
    avail = set(available_tokens)
    mapped = [(label, token) for label, token in base if token in avail]
    leftovers = sorted(avail - {t for _, t in mapped})
    mapped.extend([(tok, tok) for tok in leftovers])
    return mapped




def _summary_group_columns(domain: str, summary_token: str) -> list[str]:
    if domain == 'crops':
        mapping = {
            'province': ['Province'],
            'region': ['Region'],
            'crop_national': ['Crop'],
            'cropping_system': ['Region', 'Cropping_system'],
            'irrig_m3_class': ['Region', 'Irrig_m3_class'],
            'farm_size_class': ['Region', 'Farm_size_class'],
            'crop_group': ['Region', 'Crop_group'],
            'crop_group_national': ['Crop_group'],
        }
        return mapping.get(str(summary_token), ['Crop'])

    mapping = {
        'province': ['Province'],
        'region': ['Region'],
        'product': ['Product'],
        'national': ['Product', 'System'],
    }
    return mapping.get(str(summary_token), ['Product'])


def _current_summary_token() -> str:
    return str(crops_summary_filter.value) if domain_tbl_dd.value == 'crops' else str(livestock_summary_filter.value)


def _group_label_column(df: pd.DataFrame) -> tuple[pd.DataFrame, str, str]:
    summary_token = _current_summary_token()
    parts = [c for c in _summary_group_columns(domain_tbl_dd.value, summary_token) if c in df.columns]
    if not parts:
        parts = [c for c in ['Crop', 'Product', 'Region', 'Province'] if c in df.columns]
    if not parts:
        return df.copy(), '', 'Group'

    out = df.copy()
    gcol = '__group_label__'
    if len(parts) == 1:
        out[gcol] = out[parts[0]].astype(str)
        label = parts[0]
    else:
        out[gcol] = out[parts].astype(str).agg(' | '.join, axis=1)
        label = ' + '.join(parts)
    return out, gcol, label


def _expand_otros_rows_for_display(df: pd.DataFrame, gcol: str) -> pd.DataFrame:
    if df.empty or domain_tbl_dd.value != 'crops':
        return df
    if _current_summary_token() != 'crop_national':
        return df
    if 'Crop' not in df.columns or gcol not in df.columns:
        return df

    crop_sel_u = str(crops_crop_filter.value).strip().upper()
    if crop_sel_u not in OTROS_CROP_LABELS:
        return df

    # Expand aggregate OTROS row into OTROS sub-crops for display, matching notebook-2 expectation.
    if str(crops_subcrop_filter.value) != 'All':
        sub_targets = [str(crops_subcrop_filter.value)]
    else:
        sub_targets = _list_otros_subcrops(crop_sel_u)

    if not sub_targets:
        return df

    mask = df['Crop'].astype(str).str.upper() == crop_sel_u
    if not mask.any():
        return df

    base_rows = df.loc[mask].copy()
    other_rows = df.loc[~mask].copy()

    expanded = []
    for sub in sub_targets:
        d = base_rows.copy()
        d['Crop'] = str(sub)
        d[gcol] = str(sub)
        expanded.append(d)

    return pd.concat([other_rows] + expanded, ignore_index=True, sort=False)




def _refresh_domain_controls_visibility():
    if domain_tbl_dd.value == 'crops':
        crops_controls_box.layout.display = ''
        livestock_controls_box.layout.display = 'none'
    else:
        crops_controls_box.layout.display = 'none'
        livestock_controls_box.layout.display = ''


def _refresh_crops_selection_options(*_):
    if state_tbl['summary_df'].empty:
        return

    state_tbl['callbacks_locked'] = True
    try:
        df = state_tbl['summary_df'].copy()

        if 'Region' in df.columns:
            regions = ['All'] + list(_sorted_unique(df, 'Region'))
            cur_region = crops_region_filter.value if crops_region_filter.value in regions else 'All'
            crops_region_filter.options = regions
            crops_region_filter.value = cur_region
            if cur_region != 'All':
                df = df[df['Region'].astype(str) == cur_region]

        if 'Category' in df.columns:
            categories = ['All'] + list(_sorted_unique(df, 'Category'))
            cur_category = crops_category_filter.value if crops_category_filter.value in categories else 'All'
            crops_category_filter.options = categories
            crops_category_filter.value = cur_category
            if cur_category != 'All':
                df = df[df['Category'].astype(str) == cur_category]

        if 'Province' in df.columns:
            provinces = _sorted_unique(df, 'Province')
            cur_prov = tuple(v for v in crops_province_filter.value if v in provinces)
            crops_province_filter.options = provinces
            crops_province_filter.value = cur_prov
            if cur_prov:
                df = df[df['Province'].astype(str).isin(list(cur_prov))]

        if 'Crop' in df.columns:
            crops = ['All'] + list(_sorted_unique(df, 'Crop'))
            cur_crop = crops_crop_filter.value if crops_crop_filter.value in crops else 'All'
            crops_crop_filter.options = crops
            crops_crop_filter.value = cur_crop

            crop_label = str(crops_crop_filter.value).strip().upper()
            if crop_label in OTROS_CROP_LABELS:
                sub_opts = ['All'] + _list_otros_subcrops(crop_label)
                cur_sub = crops_subcrop_filter.value if crops_subcrop_filter.value in sub_opts else 'All'
                crops_subcrop_filter.options = sub_opts
                crops_subcrop_filter.value = cur_sub
                crops_subcrop_filter.layout.display = ''
            else:
                crops_subcrop_filter.options = ['All']
                crops_subcrop_filter.value = 'All'
                crops_subcrop_filter.layout.display = 'none'
    finally:
        state_tbl['callbacks_locked'] = False


def _refresh_livestock_selection_options(*_):
    if state_tbl['summary_df'].empty:
        return

    state_tbl['callbacks_locked'] = True
    try:
        df = state_tbl['summary_df'].copy()

        if 'Region' in df.columns:
            regions = ['All'] + list(_sorted_unique(df, 'Region'))
            cur_region = livestock_region_filter.value if livestock_region_filter.value in regions else 'All'
            livestock_region_filter.options = regions
            livestock_region_filter.value = cur_region
            if cur_region != 'All':
                df = df[df['Region'].astype(str) == cur_region]

        if 'Province' in df.columns:
            provinces = _sorted_unique(df, 'Province')
            cur_prov = tuple(v for v in livestock_province_filter.value if v in provinces)
            livestock_province_filter.options = provinces
            livestock_province_filter.value = cur_prov
            if cur_prov:
                df = df[df['Province'].astype(str).isin(list(cur_prov))]

        if 'Product' in df.columns:
            products = _sorted_unique(df, 'Product')
            cur_products = tuple(v for v in livestock_product_filter.value if v in products)
            livestock_product_filter.options = products
            livestock_product_filter.value = cur_products
            if cur_products:
                df = df[df['Product'].astype(str).isin(list(cur_products))]

        if 'Species' in df.columns:
            species = _sorted_unique(df, 'Species')
            cur_species = tuple(v for v in livestock_species_filter.value if v in species)
            livestock_species_filter.options = species
            livestock_species_filter.value = cur_species
            if cur_species:
                df = df[df['Species'].astype(str).isin(list(cur_species))]

        if 'System' in df.columns:
            systems = _sorted_unique(df, 'System')
            cur_systems = tuple(v for v in livestock_system_filter.value if v in systems)
            livestock_system_filter.options = systems
            livestock_system_filter.value = cur_systems
    finally:
        state_tbl['callbacks_locked'] = False


def _on_domain_filter_change(*_):
    if state_tbl['callbacks_locked']:
        return

    if domain_tbl_dd.value == 'crops':
        _refresh_crops_selection_options()
    else:
        _refresh_livestock_selection_options()

    _refresh_group_values()
    _render_lci_table()


def _on_crops_summary_change(*_):
    if state_tbl['strategy_sync_locked']:
        return
    state_tbl['strategy_sync_locked'] = True
    try:
        if crops_summary_filter.value is not None and crops_summary_filter.value in list(strategy_tbl_dd.options):
            strategy_tbl_dd.value = crops_summary_filter.value
        if crops_summary_filter.value is not None:
            livestock_values = [v for _, v in livestock_summary_filter.options]
            if crops_summary_filter.value in livestock_values:
                livestock_summary_filter.value = crops_summary_filter.value
    finally:
        state_tbl['strategy_sync_locked'] = False


def _on_livestock_summary_change(*_):
    if state_tbl['strategy_sync_locked']:
        return
    state_tbl['strategy_sync_locked'] = True
    try:
        if livestock_summary_filter.value is not None and livestock_summary_filter.value in list(strategy_tbl_dd.options):
            strategy_tbl_dd.value = livestock_summary_filter.value
        if livestock_summary_filter.value is not None:
            crop_values = [v for _, v in crops_summary_filter.options]
            if livestock_summary_filter.value in crop_values:
                crops_summary_filter.value = livestock_summary_filter.value
    finally:
        state_tbl['strategy_sync_locked'] = False



def _refresh_group_values(*_):
    filtered = _apply_filters(state_tbl['summary_df'])
    labeled_df, gcol, glabel = _group_label_column(filtered)
    labeled_df = _expand_otros_rows_for_display(labeled_df, gcol)
    if labeled_df.empty or not gcol or gcol not in labeled_df.columns:
        group_values_ms.options = []
        group_values_ms.value = ()
        group_values_ms.description = 'Groups:'
        return

    vals = sorted(labeled_df[gcol].dropna().astype(str).unique().tolist())
    current = tuple(v for v in group_values_ms.value if v in vals)
    group_values_ms.description = f'{glabel}:'
    if tuple(group_values_ms.options) != tuple(vals):
        group_values_ms.options = vals
    group_values_ms.value = current if current else tuple(vals)


def _refresh_lci_dataset(*_):
    with status_tbl_out:
        status_tbl_out.clear_output()
        try:
            summary_df, _, _ = load_merged(domain_tbl_dd.value, strategy_tbl_dd.value, run_id=run_tbl_dd.value)
            xml_long_df, order_map = _build_xml_long_table(domain_tbl_dd.value, strategy_tbl_dd.value, summary_df)

            state_tbl['summary_df'] = summary_df
            state_tbl['xml_long_df'] = xml_long_df
            state_tbl['order_map'] = order_map

            _refresh_domain_controls_visibility()
            if domain_tbl_dd.value == 'crops':
                _refresh_crops_selection_options()
            else:
                _refresh_livestock_selection_options()

            _refresh_group_values()

            print(f'Loaded summary rows: {len(summary_df):,}')
            print(f'Loaded XML exchange records: {len(xml_long_df):,}')
            print(f'Unique XML exchange rows: {xml_long_df["exchange_key"].nunique():,}')
        except Exception as e:
            state_tbl['summary_df'] = pd.DataFrame()
            state_tbl['xml_long_df'] = pd.DataFrame()
            state_tbl['order_map'] = {}
            state_tbl['filter_widgets'] = {}
            _reset_last_table_state()
            _refresh_domain_controls_visibility()
            group_values_ms.options = []
            group_values_ms.value = ()
            print(f'Error: {e}')

    _render_lci_table()


def _apply_filters(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    if domain_tbl_dd.value == 'crops':
        if 'Region' in out.columns and crops_region_filter.value != 'All':
            out = out[out['Region'].astype(str) == str(crops_region_filter.value)]
        if 'Category' in out.columns and crops_category_filter.value != 'All':
            out = out[out['Category'].astype(str) == str(crops_category_filter.value)]
        if 'Province' in out.columns and crops_province_filter.value:
            out = out[out['Province'].astype(str).isin(list(crops_province_filter.value))]

        crop_sel = str(crops_crop_filter.value)
        crop_sel_u = crop_sel.strip().upper()
        sub_sel = str(crops_subcrop_filter.value)

        if 'Crop' in out.columns and crop_sel != 'All':
            if crop_sel_u in OTROS_CROP_LABELS:
                subcrops = _list_otros_subcrops(crop_sel_u)
                out_crop_u = out['Crop'].astype(str).str.upper()

                if sub_sel != 'All':
                    sel_u = sub_sel.strip().upper()
                    # Use specific sub-crop when present in current summary table.
                    if (out_crop_u == sel_u).any():
                        out = out[out_crop_u == sel_u]
                    else:
                        # Keep OTROS aggregate if sub-crop rows are not materialized in this summary.
                        out = out[out_crop_u == crop_sel_u]
                elif subcrops:
                    # Expand to sub-crops only when those rows exist in this summary table.
                    sub_u = {x.strip().upper() for x in subcrops}
                    present = [x for x in sub_u if (out_crop_u == x).any()]
                    if present:
                        out = out[out_crop_u.isin(present)]
                    else:
                        out = out[out_crop_u == crop_sel_u]
                else:
                    # Fallback when sub-crop expansion is unavailable.
                    out = out[out_crop_u == crop_sel_u]
            else:
                out = out[out['Crop'].astype(str) == crop_sel]

        return out

    if 'Region' in out.columns and livestock_region_filter.value != 'All':
        out = out[out['Region'].astype(str) == str(livestock_region_filter.value)]
    if 'Province' in out.columns and livestock_province_filter.value:
        out = out[out['Province'].astype(str).isin(list(livestock_province_filter.value))]
    if 'Product' in out.columns and livestock_product_filter.value:
        out = out[out['Product'].astype(str).isin(list(livestock_product_filter.value))]
    if 'Species' in out.columns and livestock_species_filter.value:
        out = out[out['Species'].astype(str).isin(list(livestock_species_filter.value))]
    if 'System' in out.columns and livestock_system_filter.value:
        out = out[out['System'].astype(str).isin(list(livestock_system_filter.value))]
    return out



def _reset_last_table_state():
    state_tbl['last_table'] = pd.DataFrame()
    state_tbl['last_table_html'] = ''


def _copy_text_to_clipboard(text: str, html: bool = False):
    payload = json.dumps(str(text))
    mode = 'html' if html else 'text'
    mode_payload = json.dumps(mode)
    js_code = """
    (async () => {
      const txt = __PAYLOAD__;
      const mode = __MODE_PAYLOAD__;
      let copied = false;
      try {
        if (mode === 'html' && window.ClipboardItem && navigator.clipboard && navigator.clipboard.write) {
          const item = new ClipboardItem({
            'text/html': new Blob([txt], { type: 'text/html' }),
            'text/plain': new Blob([txt], { type: 'text/plain' })
          });
          await navigator.clipboard.write([item]);
          copied = true;
        }
      } catch (e) {}

      if (!copied) {
        try {
          if (navigator.clipboard && navigator.clipboard.writeText) {
            await navigator.clipboard.writeText(txt);
            copied = true;
          }
        } catch (e) {}
      }

      if (!copied) {
        try {
          const ta = document.createElement('textarea');
          ta.value = txt;
          ta.style.position = 'fixed';
          ta.style.left = '-9999px';
          document.body.appendChild(ta);
          ta.focus();
          ta.select();
          document.execCommand('copy');
          ta.remove();
          copied = true;
        } catch (e) {}
      }

      window.__espac_copy_success = copied;
    })();
    """.replace('__PAYLOAD__', payload).replace('__MODE_PAYLOAD__', mode_payload)
    display(Javascript(js_code))


def _copy_table_html(*_):
    with export_tbl_out:
        export_tbl_out.clear_output()
        html = state_tbl.get('last_table_html', '')
        if not html:
            print('No table available to copy yet.')
            return
        _copy_text_to_clipboard(html, html=True)
        print('Tried to copy rich HTML to clipboard. If paste still fails, copy the rendered Word-ready table below.')


def _copy_table_csv(*_):
    with export_tbl_out:
        export_tbl_out.clear_output()
        table = state_tbl.get('last_table', pd.DataFrame())
        if table.empty:
            print('No table available to copy yet.')
            return
        csv_txt = table.to_csv()
        _copy_text_to_clipboard(csv_txt, html=False)
        print('Tried to copy CSV to clipboard. If paste fails, copy from the Payload box below.')


def _copy_table_tsv(*_):
    with export_tbl_out:
        export_tbl_out.clear_output()
        table = state_tbl.get('last_table', pd.DataFrame())
        if table.empty:
            print('No table available to copy yet.')
            return
        _copy_text_to_clipboard(table.to_csv(sep='	'))
        print('Copied table TSV to clipboard.')


def _render_lci_table(*_):
    if state_tbl['callbacks_locked']:
        return

    with result_tbl_out:
        result_tbl_out.clear_output()

        long_df = state_tbl['xml_long_df']
        if long_df.empty:
            _reset_last_table_state()
            display(Markdown('No data available.'))
            return
        data = _apply_filters(long_df)
        data, gcol, group_label = _group_label_column(data)
        data = _expand_otros_rows_for_display(data, gcol)
        if not gcol or gcol not in data.columns:
            _reset_last_table_state()
            display(Markdown('No valid summary grouping columns available for this dataset.'))
            return
        selected_groups = list(group_values_ms.value)
        if selected_groups:
            data = data[data[gcol].astype(str).isin(selected_groups)]

        if data.empty:
            _reset_last_table_state()
            display(Markdown('No rows after current filters/selections.'))
            return

        counts_row = pd.Series(dtype=float)
        summary_for_n = _apply_filters(state_tbl['summary_df'])
        summary_for_n, summary_gcol, _ = _group_label_column(summary_for_n)
        summary_for_n = _expand_otros_rows_for_display(summary_for_n, summary_gcol)
        if selected_groups and summary_gcol and summary_gcol in summary_for_n.columns:
            summary_for_n = summary_for_n[summary_for_n[summary_gcol].astype(str).isin(selected_groups)]
        if summary_gcol in summary_for_n.columns:
            if 'count' in summary_for_n.columns:
                counts_row = summary_for_n.groupby(summary_gcol, dropna=True)['count'].sum()
            elif domain_tbl_dd.value == 'livestock':
                # Derive n from the same livestock aggregation basis used to build strategy summaries:
                # regroup province-level rows (same run) to the currently selected summary strategy.
                try:
                    summary_token_cur = _current_summary_token()
                    province_df = pd.DataFrame()
                    try:
                        province_df, _, _ = load_merged('livestock', 'province', run_id=run_tbl_dd.value)
                    except Exception:
                        province_df = pd.DataFrame()
                    if ('count' not in province_df.columns) or province_df.empty:
                        # Stable fallback: province-level livestock summary contains aggregation count.
                        p_counts = CSVS_DIR / '02_espac_livestock_lci_table_filtered__summary_province.csv'
                        if p_counts.exists():
                            province_df = pd.read_csv(p_counts)
                    province_df = _apply_filters(province_df)
                    group_cols = [c for c in _summary_group_columns('livestock', summary_token_cur) if c in province_df.columns]
                    if group_cols:
                        tmp = province_df.copy()
                        if len(group_cols) == 1:
                            tmp[summary_gcol] = tmp[group_cols[0]].astype(str)
                        else:
                            tmp[summary_gcol] = tmp[group_cols].astype(str).agg(' | '.join, axis=1)
                        if selected_groups:
                            tmp = tmp[tmp[summary_gcol].astype(str).isin(selected_groups)]
                        if 'count' in tmp.columns:
                            counts_row = tmp.groupby(summary_gcol, dropna=True)['count'].sum().astype(float)
                        else:
                            counts_row = tmp.groupby(summary_gcol, dropna=True).size().astype(float)

                        # Fill missing livestock groups (e.g., donkey/mule/horse) from province rows that
                        # include the full product list, using grouped row counts as proxy n.
                        p_full = CSVS_DIR / '02_espac_livestock_lci_table__summary_province.csv'
                        if p_full.exists():
                            full_df = pd.read_csv(p_full)
                            full_df = _apply_filters(full_df)
                            full_group_cols = [c for c in _summary_group_columns('livestock', summary_token_cur) if c in full_df.columns]
                            if full_group_cols:
                                tmp_full = full_df.copy()
                                if len(full_group_cols) == 1:
                                    tmp_full[summary_gcol] = tmp_full[full_group_cols[0]].astype(str)
                                else:
                                    tmp_full[summary_gcol] = tmp_full[full_group_cols].astype(str).agg(' | '.join, axis=1)
                                if selected_groups:
                                    tmp_full = tmp_full[tmp_full[summary_gcol].astype(str).isin(selected_groups)]
                                fill_counts = tmp_full.groupby(summary_gcol, dropna=True).size().astype(float)
                                counts_row = counts_row.combine_first(fill_counts)
                except Exception:
                    # Fallback so n is still rendered even if province reload fails.
                    counts_row = summary_for_n.groupby(summary_gcol, dropna=True).size().astype(float)
            else:
                counts_row = summary_for_n.groupby(summary_gcol, dropna=True).size().astype(float)

        agg_fn = agg_tbl_dd.value
        grouped = data.groupby(['exchange_key', gcol], dropna=True)['mean_value'].agg(agg_fn).unstack(gcol)
        grouped = grouped.dropna(axis=0, how='all').dropna(axis=1, how='all')

        if grouped.empty:
            _reset_last_table_state()
            display(Markdown('No numeric XML exchange values to aggregate.'))
            return

        order_map = state_tbl['order_map']
        ordered_idx = sorted(grouped.index, key=lambda x: order_map.get(x, (99, 999999, 999999)))
        table = grouped.loc[ordered_idx]

        table = _drop_all_zero_or_na_rows(table)
        table = _drop_systematic_zero_pesticide_rows(table)
        table = _drop_redundant_metadata_rows(table, domain_tbl_dd.value)

        ref_key = _reference_flow_key(domain_tbl_dd.value)
        seed_suffix = _seed_label_suffix_for_groupby(gcol)
        formatted_index = []
        for i in table.index:
            if i == ref_key:
                formatted_index.append(i)
                continue
            lbl = _to_verbose_label(i)
            if domain_tbl_dd.value == 'crops' and lbl.lower().startswith('seed,'):
                lbl = f"Seed [{seed_suffix}]"
            lbl = _format_emission_compartment_label(lbl)
            lbl = _final_display_label(lbl)
            formatted_index.append(lbl)
        table.index = formatted_index
        table = table.sort_index(axis=1)
        if not counts_row.empty:
            n_row = counts_row.reindex(table.columns)
            table = pd.concat([pd.DataFrame([n_row], index=['n']), table], axis=0)


        state_tbl['last_table'] = table.copy()
        display_table = _format_table_for_display(table)
        state_tbl['last_table_html'] = display_table.to_html(index=True, escape=False)

        display(Markdown('### Aggregated LCI Table'))
        display(Markdown(
            f'Rows follow XML exchange order exactly: reference flow -> inputs from nature -> inputs from technosphere -> emissions to air -> emissions to water -> emissions to soil. '
            f'Columns are `{group_label}` groups aggregated by `{agg_fn}`.'
        ))
        display(HTML(_build_scrollable_html(display_table)))


domain_tbl_dd.observe(_refresh_strategy_table_options, names='value')
domain_tbl_dd.observe(lambda *_: _refresh_domain_controls_visibility(), names='value')
strategy_tbl_dd.observe(_refresh_run_table_options, names='value')
strategy_tbl_dd.observe(_refresh_lci_dataset, names='value')
run_tbl_dd.observe(_refresh_lci_dataset, names='value')
group_values_ms.observe(_render_lci_table, names='value')
agg_tbl_dd.observe(_render_lci_table, names='value')
for _w in [crops_region_filter, crops_category_filter, crops_province_filter, crops_crop_filter, crops_subcrop_filter, livestock_region_filter, livestock_province_filter, livestock_product_filter, livestock_species_filter, livestock_system_filter]:
    _w.observe(_on_domain_filter_change, names='value')
crops_summary_filter.observe(_on_crops_summary_change, names='value')
livestock_summary_filter.observe(_on_livestock_summary_change, names='value')


copy_html_btn.on_click(_copy_table_html)
copy_csv_btn.on_click(_copy_table_csv)
copy_tsv_btn.on_click(_copy_table_tsv)

_refresh_strategy_table_options()
_refresh_lci_dataset()

crops_controls_box.children = [
    widgets.HBox([crops_region_filter, crops_category_filter, crops_summary_filter]),
    widgets.HBox([crops_province_filter, widgets.VBox([crops_crop_filter, crops_subcrop_filter])]),
]
livestock_controls_box.children = [
    widgets.HBox([livestock_region_filter, livestock_summary_filter]),
    widgets.HBox([livestock_province_filter, livestock_product_filter, livestock_species_filter, livestock_system_filter]),
]

controls_tbl = widgets.VBox([
    widgets.HBox([domain_tbl_dd, agg_tbl_dd, run_tbl_dd]),
    crops_controls_box,
    livestock_controls_box,
    group_values_ms,
    widgets.HBox([copy_html_btn, copy_csv_btn, copy_tsv_btn]),
    export_tbl_out,
])

display(Markdown('## Interactive LCI Table Explorer'))
display(controls_tbl)
display(status_tbl_out)
display(result_tbl_out)


## Interactive LCI Table Explorer

Output()

Output()